## Task #2. Brownian motion

Consider *N* small particles of mass *m* and radius *r* moving randomly, some of which collide with a large particle of mass *M* and radius *R*. Use a random walk to model the smaller particles, and conservation of momentum to model collisions with the larger particle. Animate the subsequent motion of the system.

In [1]:
import math
import random

import ipywidgets as widgets
import matplotlib.patches as patches
import matplotlib.pyplot as plt
from celluloid import Camera
from IPython.display import HTML, display
from ipywidgets import interact

In [2]:
# Challenge 2: Brownium Motion, Random Walk for the small particles

random.seed(
    0
)  # fixed on first run so the notebook is reproducible from the commandline


def Expirement(
    SmallCount,
    SmallRadius,
    SmallSpeed,
    SmallMass,
    LargeRadius,
    LargeMass,
    BoxLength,
    runTime,
    TimeSteps,
):
    BigX = [0]
    BigY = [0]
    BigSpeedAngle = 0
    BigSpeed = 0
    SmallX = []
    SmallY = []
    Time = 0
    BigXPos = 0
    BigYPos = 0

    for x in range(SmallCount):
        YouShallNotPass = True
        Count = 0
        while YouShallNotPass:
            xPos = random.uniform(-BoxLength / 2, BoxLength / 2)
            yPos = random.uniform(-BoxLength / 2, BoxLength / 2)
            if xPos**2 + yPos**2 > (SmallRadius + LargeRadius) ** 2:
                YouShallNotPass = False
            else:
                Count += 1
                if Count >= 10000:
                    YouShallNotPass = False

        SmallX.append([xPos])
        SmallY.append([yPos])

    while Time <= runTime:
        Time += TimeSteps
        for x in range(SmallCount):
            randomAngle = math.pi * 2 * random.random()
            dx = SmallSpeed * math.cos(randomAngle)
            dy = SmallSpeed * math.sin(randomAngle)
            Posx = SmallX[x][-1] + dx * TimeSteps
            Posy = SmallY[x][-1] + dy * TimeSteps

            if Posx - SmallRadius < (-BoxLength / 2):
                Posx = (-BoxLength / 2) + SmallRadius
            elif Posx + SmallRadius > (BoxLength / 2):
                Posx = (BoxLength / 2) - SmallRadius

            if Posy - SmallRadius < (-BoxLength / 2):
                Posy = (-BoxLength / 2) + SmallRadius
            elif Posy + SmallRadius > (BoxLength / 2):
                Posy = (BoxLength / 2) - SmallRadius

            distanceBetween = math.sqrt((Posx - BigXPos) ** 2 + (Posy - BigYPos) ** 2)

            if distanceBetween <= SmallRadius + LargeRadius:
                Largedx = BigSpeed * math.cos(BigSpeedAngle)
                Largedy = BigSpeed * math.sin(BigSpeedAngle)

                # Resolve velocities along the line of centers (normal) and
                # perpendicular to it (tangential): the 1D elastic collision
                # formula only applies to the normal component.
                if distanceBetween == 0:
                    # Centers exactly coincide: line of centers is undefined,
                    # so pick an arbitrary normal direction to avoid a divide-by-zero.
                    Nx, Ny = 1.0, 0.0
                else:
                    Nx = (Posx - BigXPos) / distanceBetween
                    Ny = (Posy - BigYPos) / distanceBetween
                Tx = -Ny
                Ty = Nx

                SmallNormal = dx * Nx + dy * Ny
                LargeNormal = Largedx * Nx + Largedy * Ny
                LargeTangent = Largedx * Tx + Largedy * Ty

                LargeNormalAfter = (
                    (LargeMass - SmallMass) * LargeNormal + 2 * SmallMass * SmallNormal
                ) / (LargeMass + SmallMass)

                BigXSpeed = LargeNormalAfter * Nx + LargeTangent * Tx
                BigYSpeed = LargeNormalAfter * Ny + LargeTangent * Ty

                BigSpeed = math.sqrt(BigXSpeed**2 + BigYSpeed**2)
                BigSpeedAngle = math.atan2(BigYSpeed, BigXSpeed)

                if 0 < distanceBetween < SmallRadius + LargeRadius:
                    alpha = distanceBetween / (SmallRadius + LargeRadius)
                    Posx = BigXPos + (Posx - BigXPos) / alpha
                    Posy = BigYPos + (Posy - BigYPos) / alpha

            SmallX[x].append(Posx)
            SmallY[x].append(Posy)

        BigXPos += BigSpeed * math.cos(BigSpeedAngle) * TimeSteps
        BigYPos += BigSpeed * math.sin(BigSpeedAngle) * TimeSteps

        if BigXPos - LargeRadius < (-BoxLength / 2):
            BigXPos = (-BoxLength / 2) + LargeRadius
            BigSpeedAngle = math.atan2(
                BigSpeed * math.sin(BigSpeedAngle), -BigSpeed * math.cos(BigSpeedAngle)
            )

        elif BigXPos + LargeRadius > (BoxLength / 2):
            BigXPos = (BoxLength / 2) - LargeRadius
            BigSpeedAngle = math.atan2(
                BigSpeed * math.sin(BigSpeedAngle), -BigSpeed * math.cos(BigSpeedAngle)
            )

        if BigYPos - LargeRadius < (-BoxLength / 2):
            BigYPos = (-BoxLength / 2) + LargeRadius
            BigSpeedAngle = math.atan2(
                -BigSpeed * math.sin(BigSpeedAngle), BigSpeed * math.cos(BigSpeedAngle)
            )

        elif BigYPos + LargeRadius > (BoxLength / 2):
            BigYPos = (BoxLength / 2) - LargeRadius
            BigSpeedAngle = math.atan2(
                -BigSpeed * math.sin(BigSpeedAngle), BigSpeed * math.cos(BigSpeedAngle)
            )

        BigX.append(BigXPos)
        BigY.append(BigYPos)

    fig, ax = plt.subplots(figsize=(6, 6))
    camera = Camera(fig)
    for t in range(len(BigX)):
        ax.set_xlim(-BoxLength / 2, BoxLength / 2)
        ax.set_ylim(-BoxLength / 2, BoxLength / 2)
        ax.set_aspect("equal")
        ax.grid(True)

        rect = patches.Rectangle(
            (-BoxLength / 2, -BoxLength / 2),
            BoxLength,
            BoxLength,
            fill=False,
            edgecolor="black",
            linewidth=2,
        )
        ax.add_patch(rect)

        for x in range(len(SmallX)):
            circle = patches.Circle(
                (SmallX[x][t], SmallY[x][t]), SmallRadius, color="blue", alpha=0.5
            )
            ax.add_patch(circle)

        ax.plot(BigX[: t + 1], BigY[: t + 1], color="red", linewidth=1, alpha=0.5)
        circle = patches.Circle(
            (BigX[t], BigY[t]), LargeRadius, fill=False, edgecolor="red", linewidth=2
        )
        ax.add_patch(circle)

        ax.text(
            0.02,
            0.98,
            f"Time: {round(t*TimeSteps,2)}",
            transform=ax.transAxes,
            va="top",
        )
        camera.snap()

    plt.close(fig)
    animation = camera.animate(interval=50)
    display(HTML(animation.to_jshtml()))


interact(
    Expirement,
    SmallCount=widgets.IntSlider(
        value=40, min=1, max=100, step=1, description="N (small particles)"
    ),
    SmallRadius=widgets.FloatSlider(
        value=0.3, min=0.01, max=10, step=0.01, description="r (small radius)"
    ),
    SmallSpeed=widgets.FloatSlider(
        value=2, min=0.1, max=10, step=0.01, description="Speed of Small Particles"
    ),
    SmallMass=widgets.FloatSlider(
        value=1, min=0.01, max=10, step=0.01, description="m (small mass)"
    ),
    LargeRadius=widgets.FloatSlider(
        value=3, min=1, max=100, step=0.1, description="R (large radius)"
    ),
    LargeMass=widgets.FloatSlider(
        value=50, min=10, max=100, step=0.1, description="M (large mass)"
    ),
    BoxLength=widgets.FloatSlider(
        value=25, min=5, max=500, step=1, description="Length of the Box"
    ),
    runTime=widgets.FloatSlider(
        value=20, min=10, max=5000, step=1, description="Simulation Length"
    ),
    TimeSteps=widgets.FloatSlider(
        value=0.2, min=0.1, max=5, step=0.1, description="Time Steps"
    ),
)

interactive(children=(IntSlider(value=40, description='N (small particles)', min=1), FloatSlider(value=0.3, de…

<function __main__.Expirement(SmallCount, SmallRadius, SmallSpeed, SmallMass, LargeRadius, LargeMass, BoxLength, runTime, TimeSteps)>